# Teacher Foundation Model — Training Pipeline

Baseline: `v1.0-baseline`  
Dataset: Google Drive → `MarketFoundation/storage/`

## Cell 1 — GPU Information

In [ ]:
import torch
import platform

print("=" * 60)
print("Python:", platform.python_version())
print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("=" * 60)

## Cell 2 — Clone Repository & Checkout Baseline Tag

In [ ]:
!git clone https://github.com/sandeep999-cyber/emptyu.git
%cd emptyu
!git checkout v1.0-baseline

## Cell 2b — Verify Git Revision

In [ ]:
!git rev-parse HEAD
!git describe --tags
!git status --short

## Cell 3 — Install Dependencies

In [ ]:
!pip install -r requirements.txt

## Cell 3b — CUDA Re-check (fail-fast if torch was downgraded)

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA unavailable after dependency installation. "
    "PyTorch may have been replaced by a CPU-only build."
)

print("GPU:", torch.cuda.get_device_name(0))

## Cell 4 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Cell 5 — Copy Dataset to Local SSD & Symlink Outputs to Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/MarketFoundation")
REPO = Path("/content/emptyu")

# --- Copy dataset to local SSD for performance & reliability ---
# storage/training: ~5 MB (manifests, fingerprints, DuckDB index)
# storage/canonical: ~322 MB (parquet files)
# Total ~327 MB — fits easily in Colab's ephemeral disk.

src_training = DRIVE / "storage/training"
dst_training = REPO / "storage/training"
if dst_training.exists():
    shutil.rmtree(dst_training)
shutil.copytree(src_training, dst_training)
print(f"Copied {src_training} → {dst_training}")

src_canonical = DRIVE / "storage/canonical"
dst_canonical = REPO / "storage/canonical"
if dst_canonical.exists():
    shutil.rmtree(dst_canonical)
shutil.copytree(src_canonical, dst_canonical)
print(f"Copied {src_canonical} → {dst_canonical}")

# --- Symlink persistent output dirs → Drive ---
# These are checkpointed every epoch and must survive disconnects.
for name in ["models", "logs", "evaluation"]:
    dst = REPO / name
    src = DRIVE / name
    src.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    dst.symlink_to(src)
    print(f"Symlinked {dst} → {src}")

# Verify local storage is in place
assert (REPO / "storage/training/index.duckdb").exists(), "Local training dir not copied"
assert (REPO / "storage/canonical").exists(), "Local canonical dir not copied"
print("\n✅ Data copied to local SSD; outputs symlinked to Drive")

## Cell 6 — Dataset Integrity Check

In [ ]:
import json
from pathlib import Path
import duckdb

REPO = Path("/content/emptyu")

# --- Assert fingerprint ---
fp = json.loads((REPO / "storage/training/dataset_fingerprint.json").read_text())
EXPECTED_FP = "328a7b67b070b95e47ba450452032a93dfa410431e0cf329de6a4ac7b5ae3875"
assert fp["fingerprint"] == EXPECTED_FP, f"Fingerprint mismatch: {fp['fingerprint']}"
assert fp["file_count"] == 510, f"File count mismatch: {fp['file_count']}"

# --- Assert manifest splits ---
manifest = json.loads((REPO / "storage/training/training_manifest_v1.json").read_text())
m = manifest["training_manifest"]
assert m["splits"]["train"]["symbols"] == ["BTCUSDT", "ETHUSDT"], "Train split mismatch"
assert m["splits"]["validation"]["symbols"] == ["SOLUSDT"], "Validation split mismatch"

# --- Assert DuckDB opens ---
conn = duckdb.connect(str(REPO / "storage/training/index.duckdb"))
conn.execute("SELECT count(*) FROM file_index_v1")
conn.close()

# --- Assert one parquet read succeeds ---
sample_dir = REPO / "storage/canonical/futures/BTCUSDT/klines/1m"
if sample_dir.exists():
    parquet_files = list(sample_dir.rglob("*.parquet"))
    assert len(parquet_files) > 0, "No parquet files found in sample dir"

print("✅ Dataset integrity verified")
print(f"  Fingerprint: {fp['fingerprint'][:16]}...")
print(f"  Files: {fp['file_count']}")
print(f"  Train: {m['splits']['train']['symbols']}")
print(f"  Validation: {m['splits']['validation']['symbols']}")
print(f"  Snapshot: {fp['dataset_versions']['snapshot']}")
print(f"  Alignment: {fp['dataset_versions']['alignment_version']}")
print(f"  Windowing: {fp['dataset_versions']['windowing_version']}")

## Cell 7 — Verify Snapshot

In [ ]:
from pathlib import Path

snapshot_dir = Path("storage/training/snapshots")
print(f"Snapshot dir: {snapshot_dir}")
print(f"Exists: {snapshot_dir.exists()}")

if snapshot_dir.exists():
    for p in sorted(snapshot_dir.iterdir()):
        print(f"  {p.name}")

## Cell 8 — Smoke Test

In [ ]:
!python -m src.training.train_teacher \
    --model-config configs/model_v1.yaml \
    --optimizer-config configs/optimizer_v1.yaml \
    --trainer-config configs/trainer_v1.yaml \
    --smoke

## Cell 8b — Experiment Summary

In [ ]:
import json
from pathlib import Path
import subprocess

fp = json.loads(Path("storage/training/dataset_fingerprint.json").read_text())
manifest = json.loads(Path("storage/training/training_manifest_v1.json").read_text())
m = manifest["training_manifest"]
git_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()[:8]
git_tag = subprocess.check_output(["git", "describe", "--tags"]).decode().strip()

print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Dataset fingerprint: {fp['fingerprint'][:16]}...")
print(f"Snapshot:          {fp['dataset_versions']['snapshot']}")
print(f"Train:             {m['splits']['train']['symbols']}")
print(f"Validation:        {m['splits']['validation']['symbols']}")
print(f"Model:             teacher_transformer_v1")
print(f"Objective:         Masked Market Modeling")
print(f"Commit:            {git_hash}")
print(f"Tag:               {git_tag}")
print("=" * 60)

## Cell 9 — Pilot Training

In [ ]:
print("=" * 60)
print("⚠️  Pilot training")
print()
print("Colab may disconnect before training finishes.")
print("Checkpoints are written every epoch.")
print("If disconnected, reconnect and use the Resume cell.")
print("=" * 60)

!python -m src.training.train_teacher \
    --model-config configs/model_v1.yaml \
    --optimizer-config configs/optimizer_v1.yaml \
    --trainer-config configs/trainer_v1.yaml

## Cell 10 — Resume Training (if needed)

In [ ]:
# Uncomment and fill in your run ID:
# !python -m src.training.train_teacher \
#     --model-config configs/model_v1.yaml \
#     --optimizer-config configs/optimizer_v1.yaml \
#     --trainer-config configs/trainer_v1.yaml \
#     --resume models/foundation/teacher_v1/<RUN_ID>

## Cell 11 — Find Latest Checkpoint

In [ ]:
from pathlib import Path

checkpoint_dirs = sorted(Path("models/foundation/teacher_v1").iterdir()) if Path("models/foundation/teacher_v1").exists() else []

if checkpoint_dirs:
    latest = checkpoint_dirs[-1]
    print(f"Latest checkpoint dir: {latest}")
    %env CHECKPOINT_DIR {latest}
else:
    print("No checkpoints found. Run training first.")

## Cell 12 — Clustering Evaluation

In [ ]:
!python -m src.evaluation.embedding.clustering \
    --checkpoint "$CHECKPOINT_DIR" \
    --split train \
    --pooling mean

## Cell 13 — Retrieval Evaluation

In [ ]:
!python -m src.evaluation.embedding.retrieval \
    --checkpoint "$CHECKPOINT_DIR" \
    --split validation \
    --pooling mean

## Cell 14 — Temporal Consistency

In [ ]:
!python -m src.evaluation.embedding.temporal_consistency \
    --checkpoint "$CHECKPOINT_DIR" \
    --split validation \
    --pooling mean

## Cell 15 — Linear Probe

In [ ]:
for pooling in cls mean attention; do
    echo "=== Pooling: $pooling ==="
    python -m src.evaluation.embedding.linear_probe \
        --checkpoint "$CHECKPOINT_DIR" \
        --pooling $pooling
done

## Cell 16 — Visualization

In [ ]:
!python -m src.evaluation.embedding.visualization \
    --checkpoint "$CHECKPOINT_DIR" \
    --method pca \
    --pooling mean

## Cell 17 — Archive Results & Sync DuckDB Back to Drive

In [ ]:
import shutil
from pathlib import Path

# --- Archive evaluation results ---
shutil.make_archive(
    "/content/drive/MyDrive/MarketFoundation/phase2_results",
    "zip", "evaluation"
)
print("Results archived to /content/drive/MyDrive/MarketFoundation/phase2_results.zip")

# --- Sync updated DuckDB files back to Drive ---
# These were copied locally in Cell 5 for write reliability.
DRIVE_TRAINING = Path("/content/drive/MyDrive/MarketFoundation/storage/training")
for db_name in ["index.duckdb", "experiment_registry.duckdb"]:
    local = Path("storage/training") / db_name
    remote = DRIVE_TRAINING / db_name
    if local.exists():
        shutil.copy2(local, remote)
        print(f"Synced {db_name} → Drive")

print("\n✅ Done")

## Optional — Save Environment Snapshot

In [ ]:
!pip freeze > environment.txt
!nvidia-smi